<h3>You must download and import this <a href="https://www.kaggle.com/datasets/mldatastudent/league-of-legends-match-data">Kaggle dataset</a>.

<h2>Import requirements

In [100]:
import pandas as pd
import plotly.express as px

<h2>Read in the raw data (per-player)

In [101]:
player_data = pd.read_csv("lol_match_data.csv")
print("Number of columns: ", player_data.shape[1])
print("Number of rows: ", player_data.shape[0])
player_data.head(1)

Number of columns:  108
Number of rows:  205110


,match_matchId,match_gameStartTimestamp,match_gameEndTimestamp,match_gameDuration,match_mapId,match_platformId,player_puuid,player_teamId,player_teamPosition,player_lane,...,player_item3_categories,player_item3_priceTotal,player_item4_name,player_item4_description,player_item4_categories,player_item4_priceTotal,player_item5_name,player_item5_description,player_item5_categories,player_item5_priceTotal
0,LA1_1531159804,1.720820e+12,1.720820e+12,1834.0,11.0,LA1,QPstXBo4FWSoly8yTtzmHFjsgtwUJrVzRhFTWlO3irBaEd...,blue,TOP,JUNGLE,...,"['Health', 'Damage', 'CooldownReduction', 'Abi...",3100,Sterak's Gage,<mainText><stats><attention> 400</attention> H...,"['Health', 'Damage', 'Tenacity']",3200,Stealth Ward,<mainText><stats></stats><br><br> <active>ACTI...,"['Active', 'Jungle', 'Lane', 'Trinket', 'Vision']",0


<p>This data is gathered from high-rank lobbies, which contain a small pool of re-ocurring players. Each player may appear more than once in the dataset, on different teams during different matches. Therefore, the number of rows != the number of unique players (205,110 vs. 24,279). Each game contains 10 players, so we know there are 20,511 total games included.

In [ ]:
print("Number of unique players: ", player_data["player_puuid"].nunique())
print("Number of unique teams: ", len(player_data[["match_matchId", "player_teamId"]].drop_duplicates()))
print("Number of unique matches: ", player_data["match_matchId"].nunique())

Number of unique players:  24279
Number of unique teams:  41022
Number of unique matches:  20511


<h2>Clean and aggregate the data (per-team)

In [103]:
# make new unique match-team ids to avoid needing to use both columns later
player_data["match_teamId"] = player_data["match_matchId"] + player_data["player_teamId"]

<p>We only want to keep columns that we will use as features in our model. We can remove unnecessary columns related to player accounts, perks and items. The "player_win" column will eventually be our label (this tells us whether this team won the game).

In [ ]:
features_of_interest = ["match_teamId", "match_gameDuration", "player_teamPosition", "player_win",
                          "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore",
                          "team_baron_kills", "team_dragon_kills", "team_riftHerald_kills", "team_tower_kills"]
player_data = player_data[features_of_interest]

<p>Below you can see one team, notice the five different player roles. Player-specific stats such as "player_kills" vary per row while team stats such as "player_win" and "team_dragon_kills" are the same.

In [ ]:
player_data.head(5)

,match_teamId,match_gameDuration,player_teamPosition,player_win,player_kills,player_deaths,player_assists,player_goldEarned,player_visionScore,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_tower_kills
0,LA1_1531159804blue,1834.0,TOP,True,14.0,4.0,5.0,15613.0,29.0,1.0,3.0,0.0,7.0
1,LA1_1531159804blue,1834.0,JUNGLE,True,2.0,4.0,17.0,10279.0,28.0,1.0,3.0,0.0,7.0
2,LA1_1531159804blue,1834.0,MIDDLE,True,2.0,6.0,16.0,10314.0,20.0,1.0,3.0,0.0,7.0
3,LA1_1531159804blue,1834.0,BOTTOM,True,19.0,7.0,10.0,17195.0,23.0,1.0,3.0,0.0,7.0
4,LA1_1531159804blue,1834.0,UTILITY,True,2.0,5.0,24.0,9350.0,87.0,1.0,3.0,0.0,7.0


<h4>Isolate position-specific data

In [106]:
top = player_data[player_data["player_teamPosition"] == "TOP"]
jungle = player_data[player_data["player_teamPosition"] == "JUNGLE"]
middle = player_data[player_data["player_teamPosition"] == "MIDDLE"]
bottom = player_data[player_data["player_teamPosition"] == "BOTTOM"]
support = player_data[player_data["player_teamPosition"] == "UTILITY"]

<h4>Aggregate team stats by match

In [109]:
matches_df = player_data[["match_teamId"]]
matches_df = matches_df.drop_duplicates().reset_index().drop(columns=["index"])
matches_df.head(2)

,match_teamId
0,LA1_1531159804blue
1,LA1_1531159804red


In [110]:
# save each role's stats
roles = {"top": top, "jg": jungle, "mid": middle, "bot": bottom, "sup": support}
for role in roles:
    matches_df = matches_df.merge(roles[role][["match_teamId", "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore"]], on="match_teamId", how="left"
                              ).rename(columns={"player_kills": f"{role}_kills", "player_deaths": f"{role}_deaths", "player_assists": f"{role}_assists", "player_goldEarned": f"{role}_gold", "player_visionScore": f"{role}_vision"})

# save team stats
row_per_team = player_data.drop_duplicates(subset="match_teamId")
matches_df["team_baron_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_baron_kills"])
matches_df["team_dragon_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_dragon_kills"])
matches_df["team_riftHerald_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_riftHerald_kills"])
matches_df["team_tower_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_tower_kills"])

# save match outcome
matches_df["team_win"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["player_win"])

<p>Here, we can view four teams from two different games. Can we tell just by the feature values which team has TRUE for "team_win"? Look at the "team_turrets_killed" feature.

In [ ]:
matches_df.head(4)

,match_teamId,top_kills,top_deaths,top_assists,top_gold,top_vision,jg_kills,jg_deaths,jg_assists,jg_gold,...,sup_kills,sup_deaths,sup_assists,sup_gold,sup_vision,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_tower_kills,team_win
0,LA1_1531159804blue,14.0,4.0,5.0,15613.0,29.0,2.0,4.0,17.0,10279.0,...,2.0,5.0,24.0,9350.0,87.0,1.0,3.0,0.0,7.0,True
1,LA1_1531159804red,8.0,7.0,3.0,16804.0,30.0,10.0,6.0,5.0,13346.0,...,2.0,9.0,18.0,9609.0,83.0,1.0,1.0,1.0,7.0,False
2,LA1_1531152197blue,5.0,0.0,6.0,8889.0,14.0,13.0,0.0,3.0,11402.0,...,1.0,1.0,8.0,5520.0,49.0,0.0,1.0,0.0,10.0,True
3,LA1_1531152197red,0.0,8.0,0.0,5065.0,5.0,0.0,7.0,0.0,4602.0,...,1.0,5.0,1.0,6207.0,29.0,0.0,1.0,0.0,1.0,False


<h2>Should "team_tower_kills" be included as a feature?

<h4>Let's look at the distribution of the "top_kills" feature between win and loss matches.

<p>In general, this statistic is seen as a very big indicator of who wins a game, because whoever wins top lane can easily take objectives.

In [134]:
# this shows the distribution of the "top_kills" feature between win and loss matches
# in general, this is seen as a very big indicator of who wins a game
# because whoever wins top lane can easily take objectives
fig = px.box(matches_df, x="team_win", y="top_kills")
fig.show()

<p>As shown by the graph, there is a difference for this feature between win and loss, but the median only differs by 2.

<h4>Now, let's look at the "team_tower_kills" feature between win and loss matches.

In [ ]:
fig = px.box(matches_df, x="team_win", y="team_tower_kills")
fig.show()

<p>As shown here, the median number of turrets killed for win matches is the upper fence for loss matches, meaning it is definitely a "giveaway" of match results.